# Tutorial 3 — Graph traversal via `graph_neighbors(...)`

Model a graph as an ordinary edge table, then traverse it directly in a `FROM`
clause. The graph is an **overlay on your lakehouse data**, not a separate
graph database.

See `docs/tutorials/03_graph_traversal.md` for the SQL/connector form.

In [ ]:
import shutil
import pyarrow as pa
import benostreamdb as bsdb

shutil.rmtree("/tmp/bsdb_tutorial3", ignore_errors=True)

# `create_edge_table` builds the standard edge schema (source, target,
# relation, weight) and stamps the edge metadata, which auto-configures the
# forward (source) and reverse (target) CSR graph indexes.
edges = bsdb.Table.create_edge_table("file:///tmp/bsdb_tutorial3/edges")
edges.write(pa.table({
    "source": [101, 101, 205, 309],
    "target": [205, 309, 410, 410],
    "relation": ["knows", "knows", "works_with", "works_with"],
    "weight": [1.0, 1.0, 0.5, 0.5],
}))
edges.commit()
print("is edge table:", edges.is_edge_table())
print("endpoints:", edges.edge_endpoints())

## Traverse in a `FROM` clause

`graph_neighbors` is a DataFusion table function, so it works anywhere SQL
does. The `mode` argument follows the in-memory / out-of-core `GraphMode`
pattern (`auto` | `in_memory` | `out_of_core` | `cached`).

In [ ]:
session = bsdb.Session()
session.register("edges", edges)

# Nodes within 2 hops of node 101 (node, hop, seed).
print(session.sql("SELECT * FROM graph_neighbors('edges', '101', 2, 'auto')"))

# Shortest path between two nodes (node, hop).
print(session.sql("SELECT * FROM graph_shortest_path('edges', 101, 410, 'auto')"))

# Induced subgraph edges within 2 hops (source, target).
print(session.sql("SELECT * FROM graph_subgraph('edges', '101', 2, 'auto')"))

## The programmatic API

The same traversal is available without SQL:

In [ ]:
print("neighbors of 101 within 2 hops:", list(edges.graph_neighbors(101, hops=2)))
print("shortest path 101 -> 410:", edges.shortest_path(101, 410))

## Notes

- Endpoint columns are resolved **metadata-first** (`src_col`/`dst_col`), then
  by the standard name candidates (`source`/`src`/… , `target`/`dst`/…).
- Declaring `table_type = 'edge'` (via `SET TBLPROPERTIES` or
  `set_property`) auto-configures the forward/reverse CSR indexes.
- `Session.list_graph_tables()` enumerates node/edge tables — the discovery
  primitive an MCP agent uses.